In [1]:
import pandas as pd
import utils
import string
ALPHABET_LIST = list(string.ascii_uppercase)

## Utils

In [2]:
filepath = '/root/cernbox/backups/old_database/2023_07_17/pmt_202307171723.csv'
data = pd.read_csv(filepath)
data

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,remark,created_at,updated_at,created_by,updated_by
0,110003,NaN,NaN,NaN,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2022-11-16 12:41:44,NaN,NaN
1,110005,NaN,NaN,NaN,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
2,110032,NaN,NaN,NaN,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
3,110041,NaN,NaN,NaN,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
4,110050,NaN,NaN,NaN,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10710,BB0017,NaN,NaN,EBA15,23.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10711,BB0166,NaN,NaN,EBA51,14.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10712,null01,NaN,NaN,NaN,NaN,NaN,6.776,637.0,23.5,R11187\r,164.0,NaN,NaN,NaN,2021-03-31 11:24:25,2021-03-31 09:24:25,NaN,NaN
10713,XX0018,NaN,NaN,NaN,0.0,NaN,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN


## General Checks

In [14]:
data.shape

(10714, 18)

In [3]:
data.dtypes

serial_number                object
id_pmt_block                float64
module                      float64
module_in_legacy_tilecal     object
pos_in_legacy_tilecal       float64
pos_in_phase2_tilecal       float64
beta                        float64
HV_nominal                  float64
QE                          float64
pmt_type                     object
run_number                  float64
component_status             object
current_location             object
remark                       object
created_at                   object
updated_at                   object
created_by                  float64
updated_by                  float64
dtype: object

In [24]:
null_count = data.isnull().sum()
null_count[null_count < len(data)]  #Selects columns that have non null values

serial_number                   0
module_in_legacy_tilecal     1043
pos_in_legacy_tilecal         447
beta                            0
HV_nominal                      0
QE                          10315
pmt_type                        0
run_number                  10315
component_status            10627
current_location            10388
remark                      10535
created_at                      0
updated_at                      0
dtype: int64

## Serials are valid?

### Serials have multiple regex matches

We either have serial that match the expected regex, serial that are full numerical that were made numeric with the mapping of the first 2 letters to their index in the alphabet or there is a strange value called null01

In [5]:
correct_regex = '[A-Z]{2}[0-9]{4}'
correct_match = utils.match_regex('serial_number', correct_regex, data)
number_regex = '[0-9]{6}'
number_match = utils.match_regex('serial_number', number_regex, data)
data.loc[~(correct_match | number_match)]

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,remark,created_at,updated_at,created_by,updated_by
10712,null01,NaN,NaN,NaN,NaN,NaN,6.776,637.0,23.5,R11187\r,164.0,NaN,NaN,NaN,2021-03-31 11:24:25,2021-03-31 09:24:25,NaN,NaN


### Fixing

In [6]:
def map_back_to_alphabet(serial_number: str):
    serial_list = list(serial_number)
    serial_list[0] = ALPHABET_LIST[int(serial_list[0])]
    serial_list[1] = ALPHABET_LIST[int(serial_list[1])]
    return ''.join(serial_list)

In [7]:
data.loc[number_match, 'serial_number'] = data.loc[number_match].apply(
    lambda x: map_back_to_alphabet(x['serial_number']),
    axis=1
)
data = data[data['serial_number'] != 'null01']

#### Fixed? (Yes)

In [8]:
utils.is_unique('serial_number', data)

True

In [9]:
utils.match_regex('serial_number', correct_regex, data)

True

## Beta is valid?

In [18]:
data['beta'].isnull().sum()

0

In [13]:
is_positive = utils.greater_than('beta', 0, data)
data.loc[~is_positive, 'beta'].value_counts()

beta
-1.000    1013
 0.000     258
-4.406       1
-4.817       1
Name: count, dtype: int64

## HV nominal is valid?

In [19]:
data['HV_nominal'].isnull().sum()

0

In [16]:
is_positive = utils.greater_than('HV_nominal', 0, data)
data.loc[~is_positive, 'HV_nominal'].value_counts()

HV_nominal
-1.0    13
 0.0     2
Name: count, dtype: int64

## QE is valid?

All non positive QE values are NaN

In [20]:
data['QE'].isnull().sum()

10315

In [23]:
is_positive = utils.greater_than('QE', 0, data)
data.loc[~is_positive, 'QE'].value_counts()

Series([], Name: count, dtype: int64)